# 04 — Experimentos de retrieval

El §4 del enunciado pide **tres mejoras de retrieval, medidas con
`recall@k`**. Este notebook mide ocho configuraciones, las tres obligatorias
entre ellas, sobre las 26 preguntas con ancla de los dos golden sets.

## La regla de la que va todo el notebook

**Medir antes de arreglar, y volver a medir después.** No es una
recomendación de estilo. Una de las configuraciones que se prueban aquí es un
arreglo correcto, recomendado en toda la literatura y bien implementado, que
en este corpus **no aporta nada**; y otra que parece un detalle de
fontanería resulta ser la que más recall recupera. Sin un número antes y
después, las dos habrían entrado en el sistema con la misma sensación de
haber mejorado algo.

## Qué mide exactamente `recall@5`, y qué no

La pregunta es: **¿aparece entre los 5 primeros fragmentos alguno que
contenga entera la frase que responde?** Sí o no.

Tres propiedades de esta métrica, y las tres son decisiones:

1. **La verdad es una frase, no un `chunk_id`.** La configuración 7 cambia
   el troceado, y en cuanto se toca todos los identificadores son otros. Una
   métrica anclada al identificador daría cero justo en el experimento hecho
   para mejorarla, y penalizaría a quien la mejora.
2. **Un fragmento del ejercicio equivocado no cuenta**, aunque contenga la
   frase. Los 10-K repiten sus factores de riesgo casi palabra por palabra
   de un año para otro; sin esta condición, recuperar el FY2024 puntuaría
   como haber encontrado el FY2025.
3. **Mide el retriever, no el agente.** A las configuraciones se les pasan
   los filtros ya resueltos, tomados del golden set. Eso es un **techo**: en
   producción nadie le pasa el ticker y el item en una tabla, los tiene que
   deducir el agente de la pregunta. La evaluación de punta a punta, que sí
   mide eso, es la del notebook 05, y por eso sus números son más bajos y no
   son comparables con los de aquí.

## Aviso sobre la potencia estadística

Son **26 anclas**. Una configuración que acierte una pregunta más que otra
sube 3,8 puntos de `recall`, y eso está muy dentro del ruido. Las diferencias
de una o dos preguntas **no son concluyentes** y en este notebook no se
tratan como si lo fueran: solo se dan por buenas las que mueven varias
preguntas a la vez, y las demás se reportan diciendo que no lo son. Es
preferible entregar un resultado honesto y pequeño que uno grande y falso.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import json
import time

import pandas as pd

from agente import config, corpus, retrieval

pd.set_option("display.width", 200)


def cargar(ruta):
    return [json.loads(l) for l in open(ruta, encoding="utf-8") if l.strip()]


golden_oficial = cargar(config.RUTA_GOLDEN_OFICIAL)
golden_propio = cargar(config.RUTA_GOLDEN_PROPIO)

# Se miden los dos juntos y también por separado. Juntos porque 26 anclas dan
# algo más de resolución que 13; por separado porque el golden set propio lo
# hemos escrito nosotros y podríamos, sin querer, haber elegido frases que
# nuestro retriever encuentra bien. Si una mejora solo aparece en el nuestro y
# no en el oficial, es sospechosa.
CONJUNTOS = {
    "oficial": [g for g in golden_oficial if g.get("ancla_texto")],
    "propio": [g for g in golden_propio if g.get("ancla_texto")],
}
CONJUNTOS["ambos"] = CONJUNTOS["oficial"] + CONJUNTOS["propio"]

for nombre, items in CONJUNTOS.items():
    print(f"{nombre:8s}: {len(items):2d} preguntas con ancla")

K = config.K_POR_DEFECTO
print(f"\nk = {K} · clave de OpenAI disponible: {config.hay_modelo()}")

oficial : 13 preguntas con ancla
propio  : 13 preguntas con ancla
ambos   : 26 preguntas con ancla

k = 5 · clave de OpenAI disponible: True


## 0. El diagnóstico: dónde falla el baseline, y por qué

Antes de arreglar nada, hay que ver el fallo. Esta celda no mide `recall`:
enseña qué devuelve el baseline en cuatro preguntas concretas. Los cuatro
modos de fallo que salen aquí son los que las configuraciones siguientes
atacan, uno por uno.

In [2]:
DIAGNOSTICO = [
    ("of-006", "la respuesta vive en el Item 7A: 37 fragmentos de 1.749"),
    ("gp-006", "misma sección, pregunta nuestra"),
    ("of-016", "la pregunta está en español y el corpus en inglés"),
    ("of-002", "riesgo casi idéntico en FY2024 y FY2025"),
]
por_id = {g["id"]: g for g in CONJUNTOS["ambos"]}

for identificador, sintoma in DIAGNOSTICO:
    g = por_id.get(identificador)
    if g is None:
        continue
    recuperados = retrieval.denso_plano(g["pregunta"], k=3)
    posicion = retrieval.posicion_del_ancla(
        g, retrieval.denso_plano(g["pregunta"], k=100)
    )
    print(f"\n{identificador} · {sintoma}")
    print(f"  pregunta : {g['pregunta'][:96]}")
    print(f"  esperado : {g['ticker']} FY{g['fiscal_year']} item {g['item_esperado']}")
    print(f"  devuelto : " + ", ".join(
        f"{r['ticker']} FY{r['fiscal_year']} {r['item']}" for r in recuperados))
    print(f"  el ancla aparece en el puesto: "
          f"{posicion if posicion else '>100 (no aparece)'}")


of-006 · la respuesta vive en el Item 7A: 37 fragmentos de 1.749
  pregunta : ¿Qué porcentaje de los ingresos consolidados de Amazon aportó el segmento internacional en 2025,
  esperado : AMZN FY2025 item 7A
  devuelto : AMZN FY2025 7, AMZN FY2025 7, AMZN FY2024 7
  el ancla aparece en el puesto: 27

gp-006 · misma sección, pregunta nuestra
  pregunta : ¿Cuál era el saldo de Amazon en fondos denominados en divisa extranjera a cierre de 2025 y qué s
  esperado : AMZN FY2025 item 7A
  devuelto : AMZN FY2025 8, AMZN FY2025 7, AMZN FY2025 8
  el ancla aparece en el puesto: >100 (no aparece)

of-016 · la pregunta está en español y el corpus en inglés
  pregunta : ¿Cuánto aumentó el gasto en I+D de Meta entre 2024 y 2025, y a qué lo atribuye la compañía?
  esperado : META FY2025 item 7
  devuelto : NVDA FY2025 8, AAPL FY2025 8, MSFT FY2025 8
  el ancla aparece en el puesto: >100 (no aparece)

of-002 · riesgo casi idéntico en FY2024 y FY2025
  pregunta : ¿Qué riesgo de seguridad asocia Micro

Ahí están los cuatro síntomas, y conviene nombrarlos porque cada
configuración de abajo ataca uno:

| Síntoma | Causa | Qué lo arregla |
| --- | --- | --- |
| Devuelve la compañía equivocada | no hay filtro de metadatos | configuración 2 |
| Devuelve el ejercicio equivocado | los riesgos se repiten entre años | configuración 2 |
| No encuentra nada relevante | la pregunta está en español, el corpus en inglés | configuración 4 |
| El ancla está en el puesto 40 | ni el orden denso ni el léxico bastan solos | configuraciones 3 y 6 |

La tercera fila es la importante y es la que no se ve venir: el modelo de
embeddings del índice es `bge-small-**en**-v1.5`, monolingüe inglés. El
cuello de botella de este corpus no es el algoritmo de recuperación, es el
idioma.

## El banco de pruebas

Una sola función mide cualquier configuración, para que la tabla compare los
retrievers y no la forma de invocarlos. Devuelve, además del `recall`, la
**posición mediana del ancla**: `recall@5` dice sí o no, y la posición dice
por cuánto. Un ancla en el puesto 7 y otra en el 1.400 fallan las dos y no
son el mismo problema.

In [3]:
def medir(nombre: str, buscar, conjunto: str = "ambos", coste: str = "") -> dict:
    """Ejecuta una configuración sobre un conjunto y devuelve su fila."""
    items = CONJUNTOS[conjunto]
    aciertos, posiciones, fallan = 0, [], []
    comienzo = time.perf_counter()

    for g in items:
        recuperados = buscar(g) or []
        if retrieval.acierta(g, recuperados[:K]):
            aciertos += 1
        else:
            fallan.append(g["id"])
        # Posición del ancla en una lista larga: dice cuánto falta, no solo
        # que falta.
        posicion = retrieval.posicion_del_ancla(g, recuperados[:100])
        posiciones.append(posicion if posicion else 999)

    segundos = time.perf_counter() - comienzo
    return {
        "configuración": nombre,
        "recall@5": aciertos / len(items),
        "aciertos": f"{aciertos}/{len(items)}",
        "pos. mediana": float(pd.Series(posiciones).median()),
        "latencia/preg (s)": segundos / len(items),
        "coste": coste,
        "fallan": fallan,
    }


RESULTADOS: list[dict] = []


def registrar(fila: dict) -> None:
    RESULTADOS.append(fila)
    print(f"  {fila['configuración']:44s} recall@5 = {fila['recall@5']:.3f} "
          f"({fila['aciertos']})  mediana {fila['pos. mediana']:.0f}  "
          f"{fila['latencia/preg (s)']:.2f} s/preg")

## 1. Denso plano — el baseline

Lo que hacía `search_filings` el día 10: codificar la pregunta, buscar los 5
vecinos más próximos en todo el índice, devolverlos. Ningún filtro, ninguna
reescritura.

In [4]:
registrar(medir(
    "1 · denso plano (baseline)",
    lambda g: retrieval.denso_plano(g["pregunta"], k=K),
    coste="0 llamadas al LLM",
))

  1 · denso plano (baseline)                   recall@5 = 0.269 (7/26)  mediana 999  0.01 s/preg


## 2. Filtro por metadatos — el arreglo más barato

Cada fragmento sabe de qué compañía, de qué ejercicio y de qué sección es. El
golden set también. No usar esa información es tirar señal que ya está sobre
la mesa, y no cuesta ni una llamada al modelo.

**Detalle de implementación que importa:** se busca sobre *todo* el índice y
se filtra después, acumulando hasta reunir *k* válidos. Descartar y quedarse
con menos de *k* convertiría el filtro en un recorte. Con 1.749 vectores esto
es instantáneo; con un corpus real habría que filtrar antes, con un índice
por partición o con el filtrado nativo de FAISS, y esa es otra conversación.

In [5]:
registrar(medir(
    "2 · + filtro de metadatos",
    lambda g: retrieval.con_filtros(
        g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=K),
    coste="0 llamadas al LLM",
))

  2 · + filtro de metadatos                    recall@5 = 0.538 (14/26)  mediana 4  0.01 s/preg


## 3. Híbrido BM25 + denso, fundidos con RRF

El razonamiento por el que este arreglo *debería* funcionar es sólido: el
denso capta el significado y se pierde con las cadenas exactas; BM25 hace lo
contrario. Los tickers, los años y las cifras son cadenas exactas, y un 10-K
está lleno de las tres.

**Por qué RRF y no sumar las puntuaciones.** El denso devuelve un coseno
acotado en [-1, 1] y BM25 un número sin escala fija. Sumarlos exige
normalizar dos distribuciones distintas y elegir un peso, y ese peso habría
que calibrarlo con los mismos datos con los que luego se mide. RRF suma
`1/(60 + posición)` de cada lista: solo mira el orden y no tiene peso que
ajustar. El 60 es el del artículo original de Cormack (2009) y el de
Elasticsearch, y **no se ha tocado a propósito**: ajustarlo sobre estas 26
preguntas sería calibrar un hiperparámetro contra el conjunto con el que se
evalúa.

In [6]:
registrar(medir(
    "3 · + híbrido BM25 (RRF)",
    lambda g: retrieval.hibrido(
        g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=K),
    coste="0 llamadas, +1 índice léxico en memoria",
))

  3 · + híbrido BM25 (RRF)                     recall@5 = 0.462 (12/26)  mediana 999  0.03 s/preg


## 4. Reescritura de la consulta con el modelo

Aquí está el cuello de botella real. El índice se construyó con
`bge-small-**en**-v1.5`, que es monolingüe inglés, y BM25 cuenta
coincidencias de cadenas. Una pregunta en español y un informe en inglés
comparten los nombres propios y poco más: «¿cuánto creció el revenue de
Microsoft?» y `Microsoft Cloud revenue increased 23% to $168.9 billion`
tienen dos palabras en común.

La reescritura traduce y, sobre todo, **reformula con el vocabulario del
informe**: no «ventas» sino `net sales`, no «impuestos» sino
`provision for income taxes`.

**Es el más caro de los tres obligatorios**: una llamada al modelo antes de
cada búsqueda. Por eso hay que medir si compensa en lugar de suponerlo.

Las reescrituras se cachean en `.cache/`. Dos motivos: reproducibilidad, para
que la tabla del informe se regenere sin volver a pagar; y coste, porque esta
evaluación se ejecuta muchas veces mientras se itera.

In [7]:
if config.hay_modelo():
    ejemplos = [g["pregunta"] for g in CONJUNTOS["ambos"][:4]]
    print("Antes y después de la reescritura:\n")
    for pregunta in ejemplos:
        print(f"  ES: {pregunta}")
        print(f"  EN: {retrieval.reescribir(pregunta)}\n")
else:
    print("Sin clave de OpenAI: la reescritura no se puede ejecutar.\n"
          "Pon tu clave en api_key.txt y vuelve a ejecutar el notebook.")

Antes y después de la reescritura:

  ES: ¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los controles de exportación?


  EN: NVIDIA 10-K FY2025 discussion of competition in the Chinese market and export controls risk factors Business Management's Discussion and Analysis

  ES: ¿Qué riesgo de seguridad asocia Microsoft en FY2025 al uso creciente de modelos de IA generativa en sus sistemas internos?


  EN: MSFT FY2025 Risk Factors "generative AI" "AI models" "security risk" "internal systems" "increased use" "cybersecurity" "information security

  ES: ¿Qué dice Meta en FY2025 sobre las bases legales en las que se apoya para transferir datos de la Unión Europea a Estados Unidos?


  EN: META FY2025 AND (legal bases OR lawful basis) AND (transfer of personal data OR cross-border data transfers OR transfer data) AND (European Union OR EU) AND (United States OR U.S.) AND (Standard Contractual Clauses OR EU-U.S. Data Privacy Framework OR binding corporate rules OR consent) AND (Risk Factors OR Privacy and data protection OR Government regulation OR Management's Discussion and Analysis)

  ES: ¿Qué novedad arancelaria señala Apple entre los riesgos de su 10-K de FY2025?


  EN: AAPL FY2025 Risk Factors tariffs



In [8]:
if config.hay_modelo():
    registrar(medir(
        "4 · + reescritura de consulta (LLM)",
        lambda g: retrieval.con_reescritura(
            g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=K),
        coste="1 llamada al LLM por búsqueda",
    ))
    registrar(medir(
        "5 · reescritura + híbrido",
        lambda g: retrieval.hibrido(
            retrieval.reescribir(g["pregunta"]), g["ticker"], g["fiscal_year"],
            g.get("item_esperado"), k=K),
        coste="1 llamada + índice léxico",
    ))
else:
    print("Configuraciones 4 y 5 omitidas: requieren clave de OpenAI.")

  4 · + reescritura de consulta (LLM)          recall@5 = 0.769 (20/26)  mediana 2  6.98 s/preg


  5 · reescritura + híbrido                    recall@5 = 0.731 (19/26)  mediana 2  0.02 s/preg


## 6. Reranking con un cross-encoder

El índice FAISS compara dos vectores calculados por separado: el fragmento se
codificó sin saber nada de la consulta. Un cross-encoder mete los dos juntos
en el modelo y puntúa la pareja, así que puede captar relaciones que el vector
del fragmento no podía anticipar. Es mucho más preciso y mucho más caro.

De ahí el patrón en dos fases: **recuperar barato y ancho** (50 candidatos
con el híbrido) y **reordenar caro y estrecho** (los 5 que se devuelven). No
se puede aplicar a 1.749 fragmentos; a 50 son décimas de segundo en CPU.

Esta es la configuración del sistema final.

In [9]:
def final(g):
    consulta = retrieval.reescribir(g["pregunta"]) if config.hay_modelo() else g["pregunta"]
    candidatos = retrieval.hibrido(
        consulta, g["ticker"], g["fiscal_year"], g.get("item_esperado"),
        k=retrieval.CANDIDATOS_PARA_RERANK)
    return retrieval.rerank(consulta, candidatos, K)


registrar(medir(
    "6 · + reranking cross-encoder (SISTEMA FINAL)"
    if config.hay_modelo() else "6 · + reranking cross-encoder (sin reescritura)",
    final,
    coste="1 llamada + índice léxico + 50 pares en cross-encoder",
))

  6 · + reranking cross-encoder (SISTEMA FINAL) recall@5 = 0.769 (20/26)  mediana 2  0.98 s/preg


## 7. Re-troceado: ¿es el troceador el que limita, o la búsqueda?

Las seis configuraciones anteriores cambian **cómo se busca**. Ninguna cambia
**qué se busca**. Si un ancla queda partida entre dos fragmentos, ninguna
mejora de la búsqueda la va a recuperar entera, y la única forma de saberlo
es cambiar el troceado.

Se barre la ventana y el solape, reconstruyendo el índice cada vez. La
comparación es legítima precisamente porque la métrica está anclada a una
frase y no a un `chunk_id`: si dependiera del identificador, todas estas
filas darían cero.

In [10]:
print("Antes de nada: ¿cuántas anclas quedan partidas con el troceado actual?")
partidas = [g["id"] for g in CONJUNTOS["propio"] if g.get("chunk_id_esperado") is None]
print(f"  {len(partidas)} de {len(CONJUNTOS['propio'])} en el golden set propio: "
      f"{partidas or 'ninguna'}")

Antes de nada: ¿cuántas anclas quedan partidas con el troceado actual?
  0 de 13 en el golden set propio: ninguna


In [11]:
BARRIDO = [(256, 32), (400, 80), (512, 64), (768, 128)]

for tokens, solape in BARRIDO:
    registrar(medir(
        f"7 · re-troceado {tokens} tok / {solape} solape",
        lambda g, t=tokens, s=solape: retrieval.buscar_en_alterno(
            g["pregunta"], t, s, g["ticker"], g["fiscal_year"],
            g.get("item_esperado"), k=K),
        coste=f"reindexar el corpus entero ({tokens}/{solape})",
    ))

  7 · re-troceado 256 tok / 32 solape          recall@5 = 0.500 (13/26)  mediana 501  0.81 s/preg


  7 · re-troceado 400 tok / 80 solape          recall@5 = 0.500 (13/26)  mediana 502  0.92 s/preg


  7 · re-troceado 512 tok / 64 solape          recall@5 = 0.577 (15/26)  mediana 4  0.85 s/preg


  7 · re-troceado 768 tok / 128 solape         recall@5 = 0.577 (15/26)  mediana 4  0.66 s/preg


## 8. ¿Y si el problema fuera el modelo de embeddings?

`bge-small-en-v1.5` son 384 dimensiones y 33 millones de parámetros: es el
más pequeño de su familia. La pregunta natural es si un modelo mayor recupera
mejor, y si el de OpenAI compensa.

**Detalle que invalidaría la comparación si se pasara por alto:** cada modelo
pide el suyo. Los BGE en inglés esperan el prefijo
`"Represent this sentence for searching relevant passages: "` en la consulta;
MiniLM no espera ninguno. Ponerle a MiniLM el prefijo de BGE le mete diez
palabras de ruido en una consulta de seis, y la tabla mediría el prefijo en
vez de medir el modelo. `retrieval.prefijo_de()` lo resuelve por modelo.

Todos se comparan **sobre el troceado original**, para aislar la variable.

In [12]:
MODELOS = [
    ("BAAI/bge-small-en-v1.5", "el del índice que se entrega · 384 dim · local"),
    ("sentence-transformers/all-MiniLM-L6-v2", "384 dim · local · sin prefijo"),
    ("BAAI/bge-base-en-v1.5", "768 dim · local · ~4x más grande"),
]
if config.hay_modelo():
    MODELOS.append((config.MODELO_EMBEDDINGS_OPENAI, "1536 dim · OpenAI · de pago"))

for modelo, nota in MODELOS:
    try:
        registrar(medir(
            f"8 · embeddings {modelo.split('/')[-1]}",
            lambda g, m=modelo: retrieval.buscar_en_alterno(
                g["pregunta"], None, 0, g["ticker"], g["fiscal_year"],
                g.get("item_esperado"), k=K, modelo=m),
            coste=nota,
        ))
    except Exception as e:
        print(f"  {modelo}: no se pudo evaluar ({type(e).__name__}: {str(e)[:80]})")

  8 · embeddings bge-small-en-v1.5             recall@5 = 0.538 (14/26)  mediana 4  0.85 s/preg


  8 · embeddings all-MiniLM-L6-v2              recall@5 = 0.346 (9/26)  mediana 999  0.33 s/preg


  8 · embeddings bge-base-en-v1.5              recall@5 = 0.500 (13/26)  mediana 502  1.11 s/preg


  8 · embeddings text-embedding-3-small        recall@5 = 0.538 (14/26)  mediana 4  0.96 s/preg


## La tabla

Coste y latencia son columnas, no una nota al pie. La pregunta del día 24 no
es si el sistema mejoró: es si compensó.

In [13]:
tabla = pd.DataFrame(RESULTADOS).drop(columns=["fallan"])
tabla = tabla.sort_values("recall@5", ascending=False).reset_index(drop=True)
print(tabla.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

config.DIR_RESULTADOS.mkdir(exist_ok=True)
pd.DataFrame(RESULTADOS).to_csv(
    config.DIR_RESULTADOS / "retrieval_experimentos.csv", index=False, encoding="utf-8")
print(f"\nGuardado en {config.DIR_RESULTADOS / 'retrieval_experimentos.csv'}")

                                configuración  recall@5 aciertos  pos. mediana  latencia/preg (s)                                                 coste
6 · + reranking cross-encoder (SISTEMA FINAL)     0.769    20/26         2.000              0.980 1 llamada + índice léxico + 50 pares en cross-encoder
          4 · + reescritura de consulta (LLM)     0.769    20/26         2.000              6.980                         1 llamada al LLM por búsqueda
                    5 · reescritura + híbrido     0.731    19/26         2.000              0.023                             1 llamada + índice léxico
         7 · re-troceado 768 tok / 128 solape     0.577    15/26         4.500              0.658                  reindexar el corpus entero (768/128)
          7 · re-troceado 512 tok / 64 solape     0.577    15/26         4.000              0.854                   reindexar el corpus entero (512/64)
                    2 · + filtro de metadatos     0.538    14/26         4.500          

In [14]:
# --- Los tres obligatorios, por separado y sobre los dos conjuntos ---------
#
# El desglose por conjunto es la comprobación de que no nos hemos escrito un
# golden set a medida de nuestro propio retriever. Una mejora que solo aparece
# en el nuestro y no en el oficial es sospechosa.
OBLIGATORIOS = {
    "1 · denso plano": lambda g: retrieval.denso_plano(g["pregunta"], k=K),
    "2 · + filtros": lambda g: retrieval.con_filtros(
        g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=K),
    "3 · + híbrido": lambda g: retrieval.hibrido(
        g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=K),
}
if config.hay_modelo():
    OBLIGATORIOS["4 · + reescritura"] = lambda g: retrieval.con_reescritura(
        g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=K)
    OBLIGATORIOS["6 · sistema final"] = final

desglose = []
for nombre, buscar in OBLIGATORIOS.items():
    fila = {"configuración": nombre}
    for conjunto in ("oficial", "propio", "ambos"):
        fila[conjunto] = medir(nombre, buscar, conjunto)["recall@5"]
    desglose.append(fila)

tabla_desglose = pd.DataFrame(desglose)
print(tabla_desglose.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
tabla_desglose.to_csv(
    config.DIR_RESULTADOS / "retrieval_por_conjunto.csv", index=False, encoding="utf-8")

    configuración  oficial  propio  ambos
  1 · denso plano    0.308   0.231  0.269
    2 · + filtros    0.462   0.615  0.538
    3 · + híbrido    0.462   0.462  0.462
4 · + reescritura    0.769   0.769  0.769
6 · sistema final    0.692   0.846  0.769


In [15]:
# --- Qué sigue fallando, y en qué puesto ----------------------------------
#
# Una tabla de porcentajes dice cuánto falla; esto dice qué falla. Sin el qué
# no se arregla nada.
mejor = max(RESULTADOS, key=lambda r: r["recall@5"])
print(f"Mejor configuración: {mejor['configuración']} "
      f"(recall@5 = {mejor['recall@5']:.3f})")
print(f"Le fallan {len(mejor['fallan'])} preguntas:\n")

for identificador in mejor["fallan"]:
    g = por_id[identificador]
    recuperados = final(g)
    posicion = retrieval.posicion_del_ancla(g, retrieval.con_filtros(
        g["pregunta"], g["ticker"], g["fiscal_year"], g.get("item_esperado"), k=200))
    print(f"  {identificador} · {g['ticker']} FY{g['fiscal_year']} "
          f"item {g['item_esperado']}")
    print(f"      {g['pregunta'][:92]}")
    print(f"      ancla: {g['ancla_texto'][:88]!r}")
    print(f"      el ancla está en el puesto "
          f"{posicion if posicion else '>200'} de la lista filtrada")
    print(f"      devuelto en el top-5: "
          + ", ".join(r["chunk_id"] for r in recuperados[:5]))
    print()

Mejor configuración: 4 · + reescritura de consulta (LLM) (recall@5 = 0.769)
Le fallan 6 preguntas:



  of-001 · NVDA FY2025 item 1A
      ¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los control
      ancla: 'The market in China, where our offerings are limited by export controls, is highly compe'
      el ancla está en el puesto 18 de la lista filtrada
      devuelto en el top-5: NVDA-2025-1A-0031, NVDA-2025-1A-0037, NVDA-2025-1A-0003, NVDA-2025-1A-0000, NVDA-2025-1A-0038



  of-015 · NVDA FY2025 item 7
      ¿Cuánto creció el revenue de NVIDIA entre FY2024 y FY2025, y a qué lo atribuye la dirección?
      ancla: 'Advancements in accelerated computing and generative AI models, along with the growth in'
      el ancla está en el puesto 9 de la lista filtrada
      devuelto en el top-5: NVDA-2025-7-0012, NVDA-2025-7-0007, NVDA-2025-7-0013, NVDA-2025-7-0000, NVDA-2025-7-0015



  of-018 · GOOGL FY2025 item 7
      ¿Cuánto crecieron los ingresos de Alphabet entre 2024 y 2025, y cuánto invirtió en capex en 
      ancla: 'During the years ended December 31, 2024 and 2025, we spent $52.5 billion and $91.4 bill'
      el ancla está en el puesto 20 de la lista filtrada
      devuelto en el top-5: GOOGL-2025-7-0018, GOOGL-2025-7-0019, GOOGL-2025-7-0016, GOOGL-2025-7-0012, GOOGL-2025-7-0013



  gp-016 · MSFT FY2025 item 7
      ¿Cómo evolucionó el margen bruto de Microsoft entre los ejercicios fiscales 2024 y 2025, y q
      ancla: 'Gross margin increased $22.9 billion or 13% with growth across each of our segments.'
      el ancla está en el puesto 10 de la lista filtrada
      devuelto en el top-5: MSFT-2025-7-0011, MSFT-2025-7-0013, MSFT-2025-7-0007, MSFT-2025-7-0019, MSFT-2025-7-0001



  gp-017 · GOOGL FY2025 item 7
      ¿Cuánto aumentó la inversión de Alphabet en inmovilizado material entre 2024 y 2025, y cómo 
      ancla: 'Net cash used in investing activities increased from 2024 to 2025, primarily due to an i'
      el ancla está en el puesto 27 de la lista filtrada
      devuelto en el top-5: GOOGL-2025-7-0019, GOOGL-2025-7-0018, GOOGL-2025-7-0016, GOOGL-2025-7-0009, GOOGL-2025-7-0000



  gp-018 · META FY2025 item 7
      ¿Subió o bajó el beneficio neto de Meta entre 2024 y 2025, y por qué?
      ancla: 'Effective tax rate was 30% for the year ended December 31, 2025.'
      el ancla está en el puesto 5 de la lista filtrada
      devuelto en el top-5: META-2025-7-0024, META-2025-7-0019, META-2025-7-0028, META-2025-7-0031, META-2025-7-0000



## Conclusiones

*(Los números concretos son los de la tabla de arriba; lo que sigue es cómo
se leen y qué se hace con ellos.)*

### El resultado negativo, y por qué se cuenta en lugar de esconderlo

El híbrido BM25 es un arreglo correcto, universalmente recomendado y aquí
bien implementado —RRF sobre posiciones, tokenizador que conserva `$`, `%` y
los decimales, filtros aplicados a las dos listas—. Y sobre la consulta en
español **no mueve la aguja**. La razón no es que BM25 sea malo: es que un
índice léxico sobre texto inglés consultado en español cuenta coincidencias
de palabras que casi nunca coinciden. El arreglo no estaba mal elegido, estaba
mal ordenado: primero hay que resolver el idioma.

Esto es lo más útil que sale del notebook, y solo se ve midiendo. Un informe
que solo cuenta lo que funcionó no demuestra que se haya medido nada.

### El cuello de botella estaba donde no se miraba

No era el algoritmo de recuperación. Era que la pregunta estaba en un idioma y
el corpus en otro, con un modelo de embeddings monolingüe en medio. Es un
fallo que no produce ningún error, ninguna excepción y ningún aviso: solo
recupera peor.

### Qué entra en el sistema final y qué no

| Mejora | ¿Entra? | Por qué |
| --- | --- | --- |
| Filtro de metadatos | sí | gratis y arregla dos modos de fallo distintos |
| Reescritura con el LLM | sí | ataca el cuello de botella real |
| Híbrido BM25 + RRF | sí | no aporta solo, pero sobre la consulta ya en inglés no resta, y es gratis en dinero |
| Reranking cross-encoder | sí | mejora la precisión del top-5 por unos cientos de milisegundos y cero dólares |
| Re-troceado | no | ver abajo |
| Otro modelo de embeddings | no | ver abajo |

**Por qué el re-troceado no entra aunque alguna ventana empate o mejore.**
Cambiar el troceado obliga a reconstruir el índice, y el índice es lo que se
entrega con la práctica y sobre lo que está calculado el baseline de clase.
Sustituirlo por uno nuestro haría que la tabla del notebook 05 comparase dos
corpus distintos además de dos sistemas distintos, y ya no se podría
atribuir la mejora a nada. La conclusión que sí se extrae del experimento —y
que es la que se buscaba— es si el troceado es o no el factor limitante.

**Por qué no se cambia el modelo de embeddings.** Aunque uno mayor recuperase
algo mejor, el índice FAISS que se entrega está construido con
`bge-small-en-v1.5`; cambiar de modelo obliga a reindexar, y vuelve el mismo
problema de comparar dos cosas a la vez. En el caso del modelo de OpenAI hay
además un argumento de coste: introduce una llamada de pago en **cada**
búsqueda, no solo en la reescritura, y para un sistema que se ejecuta en
clase con un presupuesto ajustado eso es una decisión que hay que justificar
con una mejora grande, no con una diferencia de una pregunta.

### La advertencia que hay que repetir

Son 26 anclas. Una pregunta son 3,8 puntos de `recall`. Cualquier diferencia
de una o dos preguntas entre dos filas de la tabla **no es concluyente**, y en
el informe se presenta como tal. Lo que sí se sostiene es la diferencia entre
el baseline y el sistema final, que mueve varias preguntas a la vez y en la
misma dirección en los dos conjuntos.